In [ ]:
import sys
import os

parent_dir = os.path.abspath('..')
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

In [ ]:
from src.context.deepthought_context import DeepthoughtExecContext

ctx = DeepthoughtExecContext()
embedding = ctx.create_multiple_embeddings(["Test"])

os.getcwd()

In [ ]:
from dataclasses import dataclass
from typing import Optional
import uuid

@dataclass
class DocumentChunk:
    id: str
    doc_title: str
    page_number: int
    chunk_id: int
    text: str
    embedding: Optional[list[float]] = None

In [ ]:
def chunk_page_text(
    text: str, 
    page_number: int, 
    doc_title: str, 
    chunk_len: int = 500, 
    overlap: int = 100
) -> list[DocumentChunk]:

    chunks = []
    start = 0
    chunk_id = 0

    while start < len(text):
        end = start + chunk_len
        chunk_text = text[start:end]
        
        chunks.append(
            DocumentChunk(
                id=str(uuid.uuid4()),
                doc_title=doc_title,
                page_number=page_number,
                chunk_id=chunk_id,
                text=chunk_text
            )
        )

        chunk_id += 1
        start = end - overlap
        if start < 0:
            start = 0

    return chunks


In [ ]:
from pypdf import PdfReader
import re
import unicodedata


def clean_text(text: str) -> str:
    allowed = r"[^a-zA-Z0-9äöüÄÖÜß.,:;?!\-()\"' ]"
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(allowed, " ", text)
    text = re.sub(r"\s+", " ", text)
    return text

def extract_text(path: str, doc_title: str) -> list[DocumentChunk]:
    reader = PdfReader(path)
    
    all_chunks = []

    for i, page in enumerate(reader.pages, start=1):
        raw = page.extract_text()
        clean = clean_text(raw)

        page_chunks = chunk_page_text(
            text=clean,
            page_number=i,
            doc_title=doc_title
        )

        all_chunks.extend(page_chunks)

    return all_chunks


In [ ]:
chunks = extract_text(
    path="../src/db/documents/Operating_Systems.pdf",
    doc_title="Operating_Systems"
)

In [ ]:
def embed_chunks(ctx, chunks: list[DocumentChunk]) -> list[DocumentChunk]:
    # Alle Texte extrahieren
    texts = [chunk.text for chunk in chunks]

    embeddings = ctx.create_multiple_embeddings(texts)

    for chunk, emb in zip(chunks, embeddings):
        chunk.embedding = emb

    return chunks

In [ ]:
chunks = embed_chunks(ctx=ctx, chunks=chunks)

In [ ]:
chunks

In [ ]:
import psycopg2
from psycopg2.extras import execute_batch
from psycopg2.extensions import connection
from pgvector.psycopg2 import register_vector

conn = psycopg2.connect(
    host="localhost",
    dbname="postgres",
    user="zerocool",
    password="supersecret"
)
register_vector(conn)



def insert_chunks(conn: connection, chunks: list[DocumentChunk]):
    with conn.cursor() as cur:
        execute_batch(
            cur,
            """
            INSERT INTO document_chunks (
                id, doc_title, page_number, chunk_id, text, embedding
            ) VALUES (%s, %s, %s, %s, %s, %s)
            """,
            [
                (
                    chunk.id,
                    chunk.doc_title,
                    chunk.page_number,
                    chunk.chunk_id,
                    chunk.text,
                    chunk.embedding  # Liste → pgvector akzeptiert list[float] direkt
                )
                for chunk in chunks
            ]
        )
    conn.commit()

def query_similar_chunks(conn: connection, query_embedding, top_k=5):
    with conn.cursor() as cur:
        cur.execute(
            """
            SELECT
                id,
                doc_title,
                page_number,
                chunk_id,
                text,
                1 - (embedding <=> (%s::vector)) AS cosine_similarity
            FROM document_chunks
            ORDER BY embedding <=> (%s::vector)
            LIMIT %s;
            """,
            (query_embedding, query_embedding, top_k)
        )

        return cur.fetchall()

In [ ]:


insert_chunks(conn=conn, chunks=chunks)

In [ ]:
conn.rollback()
query_emb = ctx.create_multiple_embeddings(["What do you know about threads?"])[0]
query_emb
results = query_similar_chunks(conn, query_emb, top_k=5)

for r in results:
    chunk_id = r[0]
    title = r[1]
    page = r[2]
    cid = r[3]
    text = r[4]
    similarity = r[5]

    print("--------------------------------------------------")
    print(f"ID: {chunk_id}")
    print(f"Dokument: {title}")
    print(f"Seite: {page}")
    print(f"Chunk: {cid}")
    print(f"Similarity: {similarity:.4f}")
    print(f"Text:\n{text}")
    print("--------------------------------------------------\n")